# Cellulase by Design - CCD / RSM notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harshini-Suresha/cellulase-by-design/blob/main/notebooks/cellulase_by_design.ipynb)

Self-contained: runs in **Colab, Jupyter, or VS Code** with only numpy / pandas / scipy / matplotlib (all preinstalled in Colab).

Data: Aspergillus awamori MK788209 cellulase CCD, Microbial Cell Factories 23:11 (2024), doi:10.1186/s12934-023-02286-w.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
print('ok')

## 1 - Published 13-run rotatable CCD

In [ ]:
ccd = pd.DataFrame([
    (6, 9.5, 6.85), (3.17, 9.5, 6.15), (8.83, 9.5, 8.12), (4, 12, 4.49),
    (6, 9.5, 6.85), (6, 9.5, 6.85), (6, 9.5, 6.85), (4, 7, 7.85),
    (6, 13.04, 6.67), (8, 12, 9.00), (8, 7, 7.00), (6, 5.96, 8.00), (6, 9.5, 6.85)],
    columns=['Substrate_g','Time_d','Cellulase_U_per_mL'])
ccd

## 2 - Code factors and fit full quadratic

In [ ]:
ALPHA = 2**0.25
def to_coded(v, lo, hi):
    c, s = (lo+hi)/2, (hi-lo)/2/ALPHA
    return (np.asarray(v, float)-c)/s
A = to_coded(ccd['Substrate_g'], ccd['Substrate_g'].min(), ccd['Substrate_g'].max())
B = to_coded(ccd['Time_d'], ccd['Time_d'].min(), ccd['Time_d'].max())
Y = ccd['Cellulase_U_per_mL'].to_numpy(float)
X = np.column_stack([np.ones(len(Y)), A, B, A*B, A*A, B*B])
names = ['1','A','B','AB','A2','B2']
beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
fit = X @ beta; resid = Y - fit
n, p = X.shape; dfe = n - p
sse = resid @ resid; sst = ((Y-Y.mean())**2).sum(); ssm = sst - sse
mse = sse/dfe; h = np.einsum('ij,jk,ik->i', X, np.linalg.inv(X.T@X), X)
press = ((resid/(1-h))**2).sum()
se = np.sqrt(np.diag(np.linalg.inv(X.T@X))*mse)
tv = beta/se; pv = 2*stats.t.sf(abs(tv), dfe)
print(f'R2={ssm/sst:.4f} adj={1-(1-ssm/sst)*(n-1)/dfe:.4f} pred={1-press/sst:.4f}')
for k,b,s,t,pp in zip(names,beta,se,tv,pv): print(f'{k:>3} {b:+.4f} se={s:.4f} t={t:+.2f} p={pp:.4f}')

## 3 - Contour surface + stationary point

In [ ]:
b0,bA,bB,bAB,bAA,bBB = beta
g = np.linspace(-ALPHA, ALPHA, 120); GA, GB = np.meshgrid(g, g)
Z = b0 + bA*GA + bB*GB + bAB*GA*GB + bAA*GA*GA + bBB*GB*GB
fig, ax = plt.subplots(figsize=(6,5))
cs = ax.contourf(GA, GB, Z, 20); fig.colorbar(cs, label='Cellulase U/mL')
ax.scatter(A, B, c='k', s=30)
ax.set(xlabel='A (coded)', ylabel='B (coded)', title='Fitted CCD surface')
plt.show()

## 4 - Best setting + 95% prediction interval

In [ ]:
P = np.column_stack([GA.ravel(), GB.ravel()]); P = P[np.hypot(*P.T) <= ALPHA+1e-9]
D = np.column_stack([np.ones(len(P)), P[:,0], P[:,1], P[:,0]*P[:,1], P[:,0]**2, P[:,1]**2])
yh = D @ beta; i = int(np.argmax(yh)); xo = P[i]
XtXi = np.linalg.inv(X.T@X); q = float(D[i] @ XtXi @ D[i])
tc = stats.t.ppf(0.975, dfe); pi = tc*np.sqrt(mse*(1+q))
print(f'coded optimum {xo.round(3)} -> {yh[i]:.2f} (95% PI {yh[i]-pi:.2f}..{yh[i]+pi:.2f})')

## 5 - Download example CSV (Colab)

In [ ]:
ccd.to_csv('/tmp/example_ccd.csv', index=False)
try:
    from google.colab import files; files.download('/tmp/example_ccd.csv')
except ImportError:
    print('not in Colab - CSV at /tmp/example_ccd.csv')